# XGBoost

In [1]:
from datetime import datetime
from scipy.sparse import load_npz
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    RocCurveDisplay
)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
# Carga del dataset
X_train = load_npz('datasets/X_train.npz')
X_test = load_npz('datasets/X_test.npz')

y_train = pd.read_csv('datasets/y_train.csv')['label']
y_test = pd.read_csv('datasets/y_test.csv')['label']

In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
)

start_train = datetime.now()
xgb_model.fit(X_train, y_train)
train_duration = datetime.now() - start_train

start_pred = datetime.now()
y_pred = xgb_model.predict(X_test)
predict_duration = datetime.now() - start_pred

#### Métricas

Se evalúa.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Legítimo', 'Phishing'], cmap='Blues')

In [ ]:
cm_xgb = confusion_matrix(y_test, y_pred)
P = cm_xgb[1, :].sum()
N = cm_xgb[0, :].sum()
TP = cm_xgb[1, 1]
TN = cm_xgb[0, 0]

TPR_xgb = TP / P
TNR_xgb = TN / N
balanced_accuracy_xgb = (TPR_xgb + TNR_xgb) / 2

precision_xgb = precision_score(y_test, y_pred, zero_division=0)
recall_xgb = recall_score(y_test, y_pred, zero_division=0)
f1_xgb = f1_score(y_test, y_pred, zero_division=0)

print(f"Exactitud balanceada:    {balanced_accuracy_xgb:.4f}")
print(f"Precisión:               {precision_xgb:.4f}")
print(f"Sensibilidad (Recall):   {TPR_xgb:.4f}")
print(f"F1-score:                {f1_xgb:.4f}")
print(f"Especificidad:           {TNR_xgb:.4f}")
print(f"Recuperación (Recall):   {recall_xgb:.4f}")
print(f"Tiempo de entrenamiento: {train_duration}")
print(f"Tiempo de predicción:    {predict_duration}")

In [ ]:
y_scores = xgb_model.predict_proba(X_test)[:, 1]

RocCurveDisplay.from_predictions(y_test, y_scores)